# Program 3: TCP Client-Server – Message System

## Aim:
Write a Python program to create a TCP client and server where the client sends a message and the server receives and displays the message.

## Implementation Note:
Both the server and client are combined into a single cell using Python's `threading` library. The server runs safely in a background thread so the client can run instantly without freezing the notebook UI.



In [ ]:
import socket
import threading
import time

def run_server(host, port):
    """Function to run the TCP Server in the background."""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
            # Allow immediate reuse of the port after shutdown
            server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            server_socket.bind((host, port))
            server_socket.listen(1)
            print(f"[Server] System online. Listening on {host}:{port}...\n")
            
            # Block until a client attempts to connect
            client_socket, client_address = server_socket.accept()
            with client_socket:
                print(f"[Server] Connection established with client: {client_address}")
                
                # Receive payload data from client
                data = client_socket.recv(1024)
                if data:
                    print(f"[Server Response] Received Message: '{data.decode('utf-8')}'")
                    # Send feedback verification back to client
                    client_socket.sendall("Acknowledgement: Message successfully logged!".encode('utf-8'))
                    
        print("[Server] Operations complete. Server shutting down.")
    except Exception as e:
        print(f"[Server Error] {e}")

def run_client(host, port):
    """Function to run the TCP Client and transmit input message."""
    print("=== TCP CLIENT INTERFACE ===")
    message_to_send = input("Enter the text message to transmit to the server: ").strip()
    
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:
            client_socket.connect((host, port))
            print(f"[Client] Transmitting packet data...")
            client_socket.sendall(message_to_send.encode('utf-8'))
            
            # Wait for data confirmation back from server
            response = client_socket.recv(1024)
            print(f"[Client Response] From Server: {response.decode('utf-8')}")
            
    except ConnectionRefusedError:
        print("[Client Error] Connection could not be established. Check server status.")
    except Exception as e:
        print(f"[Client Error] {e}")

if __name__ == "__main__":
    # Configuration parameters
    HOST = '127.0.0.1'
    PORT = 65432
    
    # 1. Start the server function inside an independent background thread
    server_thread = threading.Thread(target=run_server, args=(HOST, PORT), daemon=True)
    server_thread.start()
    
    # 2. Give the background server a brief fraction of a second to bind and run
    time.sleep(0.2)
    
    # 3. Call the client directly inside the main thread to capture user prompt input
    run_client(HOST, PORT)
